In [ ]:
! pip install langchain lanchain-openai langchain_community langgraph python-doten faiss-cpu Pypdf langchain-huggingface

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vector_store import FAISS
from langchain_core.tool import tool

from langgraph.graph import StateGraph, START
from typing import TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, BaseMessage
from langgraph.prebuilt import ToolNode, tools_condition

import os

load_dotenv()

In [ ]:
llm = ChatOpenAI(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("Grok_Api_key"),
    base_url="https://api.groq.com/openai/v1"
)

In [ ]:
# load document
loader = PyPDFLoader("")
docs = loader.load()

In [ ]:
len(docs)

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_document(docs)

In [ ]:
len(chunks)

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")
vector_store = FAISS.from_documents(chunks, embeddings)

In [ ]:
vector_store

In [ ]:
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k':4})

# LangGraph use to rag use as a tool

In [ ]:
@tool
def rag_tool(query):
    """
    Retrieve relevant information form the pdf document.
    use this tool when the use asks factual/ conceptual questions
    that might be answered from the stored documents.
    """

    result = retriever.invoke(query)

    context = [doc.page_contentg for doc in result]
    metadata = [doc.metadata for doc in result]

    return{
        'query': query,
        'context': context,
        'metadata': metadata
    }

In [ ]:
tools = [rag_tool]

llm_with_tools = llm.bind_tools(tools)

In [ ]:
from typing import Annotated
# State
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
def chat_node(state: ChatState):
    messages = state['messages']

    response = llm_with_tools.invoke(messages)

    return {"messages":[response]}

In [ ]:
tool_node = ToolNode(tools)

In [ ]:
# Graph
graph = StateGraph(ChatState)

# Node
graph.add_node('chat_node', chat_node)
graph.add_node('tool_node', tool_node)

# Edge
graph.add_edge(START, 'chat_node')
graph.add_conditional_edges('chat_node', tools_condition)
graph.add_edge('tool_node', 'chat_node')

chatbot = graph.compile()


In [ ]:
chatbot

In [ ]:
result = chatbot.invoke({
    'messages':[
        HumanMessage(
            content=(
                'using the pdf notes, explan how to find the ideal vaue of k in knn'
))]})

In [ ]:
print(result['messages'][-1].content)